# Breast Cancer Histopathology Classifier

This project uses histopathology data (e.g., cell nuclei radius, texture, perimeter, compactness, etc.) from the [Breast Cancer Wisconsin (Diagnostic) Data Set](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data/data) to predict whether breast cell samples are malignant or benign.

In [2]:
import pandas as pd
import matplotlib as plt
import sklearn as sk

## Check if data is clean

### Load data

In [4]:
cell_data = pd.read_csv('data.csv')
cell_data = cell_data.drop('Unnamed: 32', axis=1)  # there is an unnamed 32nd column with all NaN, so we remove that
cell_data

OSError: [Errno 22] Invalid argument: 'data.csv'

### Check for missing values

In [ ]:
cell_data.info()  # result: no missing values

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

### Check for duplicates

In [ ]:
cell_data.duplicated().sum()  # result: no duplicates

np.int64(0)

### Check for garbage values

Features:
a) radius (mean of distances from center to points on the perimeter)
    - cannot be negative or 0
b) texture (standard deviation of gray-scale values)
    - can be 0 but not negative
    - Low Standard Deviation: Implies uniform stain distribution. The nucleus appears smooth, meaning the chromatin is evenly dispersed—a characteristic typical of normal, healthy, or benign cells.
    - High Standard Deviation: Implies a highly varied pixel intensity. The nucleus will look grainy or blotchy, indicating uneven clumping and structural irregularities often used by pathologists and machine learning models to detect cancer, such as in breast mass biopsies.
c) perimeter
    - cannot be 0 or negative
d) area
    - cannot be 0 or negative
e) smoothness (local variation in radius lengths)
    - 0 means totally smooth, negative values not possible
    - the less smooth it is, the more likely it's malignant
f) compactness (perimeter^2 / area - 1.0)
    - cannot be 0 or negative
g) concavity (severity of concave portions of the contour)
    - can be 0 but not negative
h) concave points (number of concave portions of the contour)
    - can be 0 but not negative
i) symmetry (how closely the cell nucleus resembles its mirror image)
    - 
j) fractal dimension ("coastline approximation" - 1)
    - 

The mean, standard error and "worst" or largest (mean of the three
largest values) of these features were computed for each image,
resulting in 30 features.

In [1]:
cell_data['radius_se']

NameError: name 'cell_data' is not defined

In [41]:
cell_data.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [45]:
# check if all ids are unique
unique_id = cell_data['id'].is_unique
print(unique_id)

# convert diagnosis to 0s and 1s (1 = malignant, 0 = benign)
cell_data['diagnosis'] = cell_data['diagnosis'].replace({'M': 1, 'B': 0})
cell_data.tail()

True


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
564,926424,1,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,926682,1,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,926954,1,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,927241,1,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400
568,92751,0,7.76,24.54,47.92,181.0,0.05263,0.04362,0.00000,0.00000,...,9.456,30.37,59.16,268.6,0.08996,0.06444,0.0000,0.0000,0.2871,0.07039


## Exploratory Data Analysis

Can this data actually be used to predict whether cells are malignant or benign?

In [ ]:
# TODO do features actually separate benign cells from malignant cells?

# plot histograms for each feature's mean, one histogram for malignant cells and another for benign cells


# TODO are there any interactions between combos of features?
# TODO confusion matrix